In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../dataset/train.csv")

df.head()

,patient_id,site_id,triage_nurse_id,arrival_mode,arrival_hour,arrival_day,arrival_month,arrival_season,shift,age,...,gcs_total,pain_score,weight_kg,height_cm,bmi,shock_index,news2_score,disposition,ed_los_hours,triage_acuity
0,TG-UXRGA9UCO,SITE-TMP-01,NURSE-0033,walk-in,6,Monday,5,spring,morning,43,...,14,7,52.3,165.4,19.1,0.725,8,discharged,7.35,2
1,TG-B19DBBS2G,SITE-HEL-01,NURSE-0001,walk-in,6,Thursday,4,spring,morning,72,...,15,-1,73.3,164.4,27.1,0.739,1,discharged,0.70,5
2,TG-GZ97W7M6V,SITE-HEL-02,NURSE-0005,walk-in,8,Saturday,4,spring,morning,82,...,15,3,77.1,183.7,22.8,0.798,2,discharged,0.63,5
3,TG-THIB2TN9Q,SITE-HEL-02,NURSE-0026,police,7,Sunday,3,spring,morning,50,...,15,7,49.6,172.6,16.6,0.812,2,discharged,1.99,3
4,TG-J3U3LQ2QY,SITE-HEL-02,NURSE-0044,walk-in,5,Tuesday,5,spring,night,62,...,15,4,71.9,173.4,23.9,0.812,2,transferred,3.58,3


In [4]:
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

y = df['triage_acuity'] - 1
y = y.astype("int")
leaky = ['patient_id', 'triage_nurse_id', 'ed_los_hours', 'disposition']
X = df.drop(columns=['triage_acuity'] + leaky)
X = X.select_dtypes(include='number')

# cat_cols = X.select_dtypes(exclude='number').columns.tolist()   # all low-cardinality now
# X = pd.get_dummies(X, columns=cat_cols, drop_first=True).astype('float32')
tf.keras.utils.set_random_seed(42)   # before building the model

X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.1, stratify=y, random_state=42)

imp = SimpleImputer(strategy='median', add_indicator=True).fit(X_tr)
X_tr_i, X_val_i = imp.transform(X_tr), imp.transform(X_val)

scaler = StandardScaler().fit(X_tr_i)           # softmax only
X_tr_s, X_val_s = scaler.transform(X_tr_i), scaler.transform(X_val_i)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

lr = LogisticRegression(max_iter=1000)    # multinomial (softmax) by default
lr.fit(X_tr_s, y_tr)                      # use the scaled arrays
pred = lr.predict(X_val_s)
print(classification_report(y_val, pred))
print(confusion_matrix(y_val, pred))

              precision    recall  f1-score   support

           0       0.94      0.92      0.93       322
           1       0.96      0.96      0.96      1344
           2       0.85      0.84      0.84      2892
           3       0.66      0.71      0.69      2302
           4       0.71      0.64      0.68      1140

    accuracy                           0.80      8000
   macro avg       0.82      0.81      0.82      8000
weighted avg       0.80      0.80      0.80      8000

[[ 295   27    0    0    0]
 [  20 1285   39    0    0]
 [   0   25 2419  429   19]
 [   0    0  383 1638  281]
 [   0    0    6  400  734]]


In [6]:
import uuid
import joblib
joblib.dump(lr, f"models/{uuid.uuid1()}.keras")

['models/4747daf4-e7d0-11f1-9f18-c281109b357e.keras']